# classifier

In [1]:
import time as t
import os
import numpy as np
from ultralytics import YOLO
from glob import glob
import cv2
import json
import yaml
import tqdm as tqdm
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import tqdm 


from yolo_cam.eigen_cam import EigenCAM
from yolo_cam.utils.image import show_cam_on_image, scale_cam_image

## config 

In [2]:
dataset_dir = "../datasets/"
dataset_path = Path(dataset_dir)
cls_dataset_dir = dataset_path / "AllSpecies-cls"
cls_background_dataset_dir = dataset_path / "AllSpecies-cls-background"
groups = ["Coleoptera", "Hymenoptera", "Lepidoptera", "background"]
config_file = "yolo26n-cls.pt"
config_bg_file = "config_bg.yaml"


# label preparation

In [9]:
label_dict = {name:list(dataset_path.rglob(f"{name}/**/*.png")) + list(dataset_path.rglob(f"{name}/**/*.jpg")) for name in groups}
for x in label_dict.items():
    print(f"{x[0]}: {len(x[1])} images")

all_test_images  =[img for img in cls_dataset_dir.rglob("**/*.png") if "test" in img.parts] + [img for img in cls_dataset_dir.rglob("**/*.jpg") if "test" in img.parts]
print(f"Total test images: {len(all_test_images)}")

all_test_bg_images  =[img for img in cls_background_dataset_dir.rglob("**/*.png") if "test" in img.parts] + [img for img in cls_background_dataset_dir.rglob("**/*.jpg") if "test" in img.parts]
print(f"Total test background images: {len(all_test_bg_images)}")

Coleoptera: 2508 images
Hymenoptera: 576 images
Lepidoptera: 1158 images
background: 1413 images
Total test images: 145
Total test background images: 291


## YOLO loading

In [4]:
model = YOLO(config_file)

# to load a model from a previous training run, use the path to the best.pt file in the runs/train/exp directory
# model = YOLO("runs/train/exp/weights/best.pt")


# training

In [5]:
results = model.train(data=cls_dataset_dir, epochs=10, imgsz=640)

New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.54 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7805MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../datasets/AllSpecies-cls, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_sca

## evaluation metrics

In [15]:
# model loading if no training
model = YOLO("./runs/classify/train/weights/best.pt")

y_true = []
y_pred = []

misclassified_images = []

for img in tqdm.tqdm(all_test_images):
    true_label = img.parts[-2]  # Assuming the folder name is the class label
    y_true.append(true_label)
    
    result = model(str(img), conf=0.25)[0]
    
    y_pred.append(groups[result.probs.data.argmax().item()])  # Get the predicted class index

    if true_label != y_pred[-1]:
        misclassified_images.append((img, true_label, y_pred[-1]))

  0%|          | 0/145 [00:00<?, ?it/s]


image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls/test/Coleoptera/IMG_0313_specimen_3_TREOBT_NEON.BET.D20.002080.png: 640x640 Coleoptera 1.00, Hymenoptera 0.00, Lepidoptera 0.00, 1.3ms
Speed: 4.8ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls/test/Coleoptera/IMG_0291_specimen_3_MECKON_NEON.BET.D20.001719.png: 640x640 Coleoptera 1.00, Lepidoptera 0.00, Hymenoptera 0.00, 1.3ms
Speed: 5.7ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls/test/Coleoptera/IMG_0193_specimen_4_MECRUF_NEON.BET.D20.000047.png: 640x640 Coleoptera 1.00, Lepidoptera 0.00, Hymenoptera 0.00, 1.3ms
Speed: 5.8ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 

  4%|▍         | 6/145 [00:00<00:02, 57.33it/s]


image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls/test/Coleoptera/IMG_0538_specimen_2_TREOBT_NEON.BET.D20.002352.png: 640x640 Coleoptera 1.00, Lepidoptera 0.00, Hymenoptera 0.00, 1.4ms
Speed: 5.3ms preprocess, 1.4ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls/test/Coleoptera/IMG_0560_specimen_2_TREOBT_NEON.BET.D20.001770.png: 640x640 Coleoptera 1.00, Lepidoptera 0.00, Hymenoptera 0.00, 1.3ms
Speed: 4.4ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls/test/Coleoptera/IMG_0241_specimen_3_TREOBT_NEON.BET.D20.001573.png: 640x640 Coleoptera 1.00, Lepidoptera 0.00, Hymenoptera 0.00, 1.2ms
Speed: 5.3ms preprocess, 1.2ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 

 11%|█         | 16/145 [00:00<00:01, 78.25it/s]

...

 98%|█████████▊| 142/145 [00:06<00:00, 11.81it/s]


image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls/test/Hymenoptera/Colletes_anchusae_M_001.jpg: 640x640 Hymenoptera 1.00, Coleoptera 0.00, Lepidoptera 0.00, 1.3ms
Speed: 46.7ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls/test/Hymenoptera/Dufourea_iris_F_001.jpg: 640x640 Hymenoptera 1.00, Coleoptera 0.00, Lepidoptera 0.00, 1.4ms
Speed: 17.5ms preprocess, 1.4ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)


 99%|█████████▉| 144/145 [00:06<00:00, 13.13it/s]


image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls/test/Hymenoptera/Dasypoda_pyrotrichia_F_001.jpg: 640x640 Hymenoptera 1.00, Coleoptera 0.00, Lepidoptera 0.00, 1.4ms
Speed: 11.6ms preprocess, 1.4ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)


100%|██████████| 145/145 [00:06<00:00, 21.69it/s]


In [16]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot().figure_.savefig('results/cls/confusion_matrix.png')
disp.plot()

## attention

In [18]:
# model = YOLO("./runs/classify/train/weights/best.pt")
model = model.cpu()

target_layers = [model.model.model[-3]]

img = cv2.imread('images/lepidoptera.jpg')
img = cv2.resize(img, (640, 640))
rgb_img = img.copy()
img = np.float32(img) / 255

cam = EigenCAM(model, target_layers, task='cls')
grayscale_cam = cam(rgb_img)[0, :, :]
cam_image = show_cam_on_image(img, grayscale_cam, use_rgb=True)
plt.imshow(cam_image)
plt.savefig('results/cls/cam.png')
plt.show()


0: 640x640 Lepidoptera 1.00, Coleoptera 0.00, Hymenoptera 0.00, 27.9ms
Speed: 2.3ms preprocess, 27.9ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)


<Figure size 640x480 with 1 Axes>

# loading with background

In [4]:
model_bg = YOLO(config_bg_file)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
config_bg summary: 86 layers, 1,536,228 parameters, 1,536,228 gradients, 3.3 GFLOPs


# training with background

In [5]:
results = model_bg.train(data=cls_background_dataset_dir, epochs=10, imgsz=640)

New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.54 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7805MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../datasets/AllSpecies-cls-background, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=config_bg.yaml, momentum=0.937, mosaic=1.0

config_bg summary: 86 layers, 1,536,228 parameters, 1,536,228 gradients, 3.3 GFLOPs
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5865.0±2869.7 MB/s, size: 199.3 KB)
train: Scanning /home/tombellivier/Documents/CV/CV-for-GRIT/models/datasets/AllSpecies-cls-background/train... 2258 images, 0 corrupt: 100% ━━━━━━━━━━━━ 2258/2258 861.0Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4544.3±2448.2 MB/s, size: 190.6 KB)
val: Scanning /home/tombellivier/Documents/CV/CV-for-GRIT/models/datasets/AllSpecies-cls-background/val... 280 images, 0 corrupt: 100% ━━━━━━━━━━━━ 280/280 36.7Mit/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.00125, momentum=0.9) with parameter groups 39 weight(decay=0.0), 40 weight(decay=0.0005), 40 bias(decay=0.0)
Image sizes 640 train, 640 val
Using

## metrics

In [11]:
# model loading if no training
# model_bg = YOLO("./runs/classify/train-2/weights/best.pt")

y_true = []
y_pred = []

misclassified_images = []

for img in tqdm.tqdm(all_test_bg_images):
    true_label = img.parts[-2]  # Assuming the folder name is the class label
    y_true.append(true_label)

    
    result = model_bg(str(img), conf=0.25)[0]
    
    y_pred.append(groups[result.probs.data.argmax().item()])  # Get the predicted class index

    if true_label != y_pred[-1]:
        misclassified_images.append((img, true_label, y_pred[-1]))

  0%|          | 0/291 [00:00<?, ?it/s]


image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls-background/test/background/IMG_0245_specimen_1_AGOMUE_NEON.BET.D20.003085.png: 640x640 background 1.00, Lepidoptera 0.00, Coleoptera 0.00, Hymenoptera 0.00, 1.3ms
Speed: 7.4ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls-background/test/background/IMG_0233_specimen_2_TREOBT_NEON.BET.D20.001477.png: 640x640 background 1.00, Lepidoptera 0.00, Coleoptera 0.00, Hymenoptera 0.00, 1.3ms
Speed: 5.8ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls-background/test/background/IMG_0546_specimen_1_TREOBT_NEON.BET.D20.001621.png: 640x640 background 1.00, Coleoptera 0.00, Lepidoptera 0.00, Hymenoptera 0.00, 1.3ms
Speed: 4.5ms prepro

  2%|▏         | 7/291 [00:00<00:04, 69.31it/s]


image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls-background/test/background/IMG_0344_specimen_1_TREOBT_NEON.BET.D20.002459.png: 640x640 background 1.00, Coleoptera 0.00, Lepidoptera 0.00, Hymenoptera 0.00, 1.3ms
Speed: 5.5ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls-background/test/background/IMG_0246_specimen_1_AGOMUE_NEON.BET.D20.003167.png: 640x640 background 0.95, Coleoptera 0.04, Lepidoptera 0.01, Hymenoptera 0.00, 1.3ms
Speed: 5.9ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls-background/test/background/IMG_0183_specimen_4_TREOBT_NEON.BET.D20.002006.png: 640x640 background 1.00, Coleoptera 0.00, Lepidoptera 0.00, Hymenoptera 0.00, 1.3ms
Speed: 5.8ms prepro

  6%|▌         | 17/291 [00:00<00:03, 85.81it/s]

...

 99%|█████████▉| 288/291 [00:13<00:00, 12.56it/s]


image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls-background/test/Hymenoptera/Colletes_anchusae_M_001.jpg: 640x640 Hymenoptera 0.75, Lepidoptera 0.25, Coleoptera 0.00, background 0.00, 1.3ms
Speed: 44.2ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls-background/test/Hymenoptera/Dufourea_iris_F_001.jpg: 640x640 Hymenoptera 0.79, Lepidoptera 0.20, Coleoptera 0.00, background 0.00, 1.3ms
Speed: 17.7ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)


100%|█████████▉| 290/291 [00:13<00:00, 13.67it/s]


image 1/1 /home/tombellivier/Documents/CV/CV-for-GRIT/models/classifier/../datasets/AllSpecies-cls-background/test/Hymenoptera/Dasypoda_pyrotrichia_F_001.jpg: 640x640 Hymenoptera 0.85, Lepidoptera 0.15, Coleoptera 0.00, background 0.00, 1.3ms
Speed: 11.5ms preprocess, 1.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)


100%|██████████| 291/291 [00:13<00:00, 21.95it/s]


In [12]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot().figure_.savefig('results/cls_bg/confusion_matrix.png')
disp.plot()

# attention with background

In [14]:
# model_bg = YOLO("./runs/classify/train/weights/best.pt")
model_bg = model_bg.cpu()

target_layers = [model_bg.model.model[-3]]

img = cv2.imread('images/lepidoptera.jpg')
img = cv2.resize(img, (640, 640))
rgb_img = img.copy()
img = np.float32(img) / 255

cam = EigenCAM(model_bg, target_layers, task='cls')
grayscale_cam = cam(rgb_img)[0, :, :]
cam_image = show_cam_on_image(img, grayscale_cam, use_rgb=True)
plt.imshow(cam_image)
plt.savefig('results/cls_bg/cam.png')
plt.show()

0: 640x640 Lepidoptera 0.99, Hymenoptera 0.01, Coleoptera 0.00, background 0.00, 30.9ms
Speed: 2.4ms preprocess, 30.9ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)


<Figure size 640x480 with 1 Axes>